# 02 — Feature Engineering & Leakage Checks

Builds the model-ready feature matrix via `po_delay.features.build_features` and demonstrates the leakage guard that keeps post-event columns (`actual_delivery_date`, `delay_days`, `is_late`) out of the inputs.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from po_delay import config
from po_delay.data_generation import GeneratorParams, generate
from po_delay.features import FEATURE_COLUMNS, assert_no_leakage, build_features

df = generate(GeneratorParams(n_orders=20_000, n_suppliers=100, seed=42))
X, y = build_features(df)
X.shape, y.mean()

((20000, 21), np.float64(0.21045))

## Leakage guard
`assert_no_leakage` raises if a post-event column ever ends up in the feature list. Demonstrating the guard actually fires on a deliberately-corrupted list:

In [2]:
try:
    assert_no_leakage(FEATURE_COLUMNS + ["is_late"])
except ValueError as e:
    print("Caught as expected:", e)

assert not (set(X.columns) & set(config.LEAKAGE_COLUMNS)), "leakage column found in X!"

Caught as expected: Leakage columns present in feature set: ['is_late']


## Feature matrix preview

In [3]:
X.dtypes

category                             category
supplier_country                     category
incoterms                            category
shipping_mode                        category
destination_region                   category
requested_lead_time_days                int64
quantity                                int64
unit_price                            float64
total_value                           float64
payment_terms_days                      int64
is_rush_order                           int64
price_dev_from_supplier_avg           float64
supplier_open_po_count                  int64
supplier_on_time_rate_trailing365     float64
supplier_mean_delay_trailing365       float64
supplier_n_orders_trailing365           int64
ack_month                               int32
ack_day_of_week                         int32
is_quarter_end                          int64
is_holiday_period                       int64
is_new_supplier                         int64
dtype: object

In [4]:
X.head()

,category,supplier_country,incoterms,shipping_mode,destination_region,requested_lead_time_days,quantity,unit_price,total_value,payment_terms_days,...,price_dev_from_supplier_avg,supplier_open_po_count,supplier_on_time_rate_trailing365,supplier_mean_delay_trailing365,supplier_n_orders_trailing365,ack_month,ack_day_of_week,is_quarter_end,is_holiday_period,is_new_supplier
0,MRO,BR,FOB,Rail,NA-West,6,129,235.29,30351.87,90,...,0.0,0,0.5,0.0,0,1,6,0,1,1
1,Custom Fabrication,MX,CIF,Air,LATAM,24,410,612.38,251076.14,30,...,0.0,2,0.5,0.0,0,1,1,0,1,1
2,Chemicals,CN,DDP,Air,EU-West,6,484,37.89,18341.13,60,...,0.0,0,0.5,0.0,0,1,2,0,1,1
3,Electronics,BR,FOB,Road,EU-East,25,217,281.57,61100.31,60,...,0.0,0,0.5,0.0,0,1,1,0,1,1
4,MRO,MX,EXW,Road,APAC,8,135,147.24,19877.01,45,...,0.0,0,0.5,0.0,0,1,6,0,1,1


## Cold-start suppliers
Rows with no prior trailing history are flagged via `is_new_supplier` rather than silently imputed away.

In [5]:
X["is_new_supplier"].value_counts()

is_new_supplier
0    19593
1      407
Name: count, dtype: int64